# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neelamobaid/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


### Lane 2 — Content Refresh / Opportunity Scoring

- **Grain:** One row represents one content page for one client on one reporting date. The combination of `report_date`, `client_hash_id`, and `content_hash_id` is unique in the March 2026 slice.
- **Tables:** `fact_content_daily_performance` for daily page performance, with `dim_content` and `dim_clients` used for page and client context where needed.
- **Time window:** March 1–31, 2026 is the development slice. The available history differs by client, so source availability flags must be checked before using measurements.
- **Target/proxy:** A historical review-candidate proxy based on a decline in recent performance compared with a previous period. It is a prioritisation signal, not a verified editorial judgement.
- **Deliberate exclusion:** Client and content IDs are kept for grouping and tracking, not used as predictive features.


In [ ]:
!git clone https://github.com/neelamobaid/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship

print("=== Skills README ===")
!cat skills/README.md

print("\n=== Writing Data Contracts Skill ===")
!cat skills/writing-data-contracts/SKILL.md

print("\n=== FlyRank Data Skill ===")
!cat skills/flyrank/flyrank-data/SKILL.md

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 129, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 129 (delta 40), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (129/129), 1.86 MiB | 15.21 MiB/s, done.
Resolving deltas: 100% (40/40), done.
/content/flyrank-ml-internship
=== Skills README ===
# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, an

In [ ]:
print("=== Querying Big Datasets Skill ===")
!cat skills/querying-big-datasets/SKILL.md

=== Querying Big Datasets Skill ===
---
name: querying-big-datasets
description: Works with datasets far too big to download or load in pandas — SQL over remote Parquet with DuckDB, aggregate-then-model, iterate on samples. Use when a dataset has millions of rows, lives on a remote host (hf:// or s3), or a notebook runs out of memory or hits rate limits.
---

# Querying big datasets

The trick that makes 79 million rows feel small: **never bring the rows to you — send the
question to the rows.** Aggregate in SQL, bring back only the small answer, model on that.

## The pattern

```python
%pip -q install duckdb
import duckdb
con = duckdb.connect()
# remote Parquet reads like a local table — match the path to how the table ships:
# most tables are ONE flat file at the repo root:
REL = "read_parquet('hf://datasets/<org>/<name>/<table>.parquet')"
# partitioned tables are a directory of month folders:
REL = "read_parquet('hf://datasets/<org>/<name>/<table>/month=2026-03/*.parquet')"
# sever

In [ ]:
%pip -q install duckdb huggingface_hub

In [2]:
from google.colab import userdata
import duckdb

hf_token = userdata.get("HF_TOKEN")
con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

safe_token = hf_token.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [ ]:
# Check that we can read the warehouse table metadata
warehouse = "hf://datasets/FlyRank/internship-warehouse"

con.execute(f"""
    SELECT *
    FROM read_parquet('{warehouse}/dim_clients.parquet')
    LIMIT 5
""").df()

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


In [ ]:
warehouse = "hf://datasets/FlyRank/internship-warehouse"

march_path = f"{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet"

con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet('{march_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,earliest_date,latest_date
0,9841378,2026-03-01,2026-03-31


In [ ]:
result = con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date
    FROM read_parquet('{march_path}')
""").fetchone()

print("Total rows:", result[0])
print("Earliest date:", result[1])
print("Latest date:", result[2])

Total rows: 9841378
Earliest date: 2026-03-01
Latest date: 2026-03-31


In [ ]:
con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{march_path}')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [ ]:

print("=== dim_content ===")
print(con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
    )
""").df().to_string(index=False))

print("\n=== fact_content_query_90d ===")
print(con.execute("""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet'
    )
""").df().to_string(index=False))

=== dim_content ===
               column_name column_type null  key default extra
            client_hash_id     VARCHAR  YES None    None  None
           content_hash_id     VARCHAR  YES None    None  None
           keyword_hash_id     VARCHAR  YES None    None  None
               url_hash_id     VARCHAR  YES None    None  None
        keyword_char_count      BIGINT  YES None    None  None
       keyword_token_count      BIGINT  YES None    None  None
            url_char_count      BIGINT  YES None    None  None
      content_created_date        DATE  YES None    None  None
      content_updated_date        DATE  YES None    None  None
              content_type     VARCHAR  YES None    None  None
             search_volume      BIGINT  YES None    None  None
               competition      DOUBLE  YES None    None  None
         competition_level     VARCHAR  YES None    None  None
                       cpc      DOUBLE  YES None    None  None
               main_intent     VARC

In [ ]:
query_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_query_90d.parquet"
)

con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(window_start) AS earliest_window_start,
        MAX(window_start) AS latest_window_start,
        MIN(window_end) AS earliest_window_end,
        MAX(window_end) AS latest_window_end,
        COUNT(DISTINCT window_start) AS distinct_window_starts,
        COUNT(DISTINCT window_end) AS distinct_window_ends
    FROM read_parquet('{query_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,earliest_window_start,latest_window_start,earliest_window_end,latest_window_end,distinct_window_starts,distinct_window_ends
0,2414248,2026-04-02,2026-04-02,2026-06-30,2026-06-30,1,1


In [4]:
warehouse = "hf://datasets/FlyRank/internship-warehouse"

march_path = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

print("March data path configured.")

March data path configured.


In [5]:
con.execute(f"""
    SELECT
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT report_date) AS distinct_dates,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_pages
    FROM read_parquet('{march_path}')
    WHERE gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date,distinct_dates,clients,content_pages
0,2026-03-01,2026-03-31,31,47,176738


In [6]:
feb_path = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=2026-01/*.parquet"
)

feb_result = con.execute(f"""
    SELECT
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT report_date) AS distinct_dates,
        COUNT(DISTINCT client_hash_id) AS clients
    FROM read_parquet('{feb_path}')
    WHERE report_date BETWEEN DATE '2026-01-30' AND DATE '2026-01-31'
      AND gsc_data_available IS TRUE
""").df()

feb_result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date,distinct_dates,clients
0,2026-01-30,2026-01-31,2,33


In [7]:
feb_path = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=2026-02/*.parquet"
)

con.execute(f"""
    SELECT
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT report_date) AS distinct_dates,
        COUNT(DISTINCT client_hash_id) AS clients
    FROM read_parquet('{feb_path}')
    WHERE gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,earliest_date,latest_date,distinct_dates,clients
0,2026-02-01,2026-02-28,28,46


In [8]:
jan_feb_path = (
    f"{warehouse}/fact_content_daily_performance/"
    "month=2026-01/*.parquet"
)

con.execute(f"""
    SELECT
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_pages,
        COUNT(DISTINCT (client_hash_id, content_hash_id)) AS client_page_pairs
    FROM read_parquet('{jan_feb_path}')
    WHERE report_date BETWEEN DATE '2026-01-30' AND DATE '2026-01-31'
      AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,clients,content_pages,client_page_pairs
0,33,93203,93203


In [9]:
con.execute(f"""
    SELECT
        report_date,
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows
    FROM read_parquet('{march_path}')
    GROUP BY report_date
    ORDER BY report_date
""").df()

,report_date,total_rows,gsc_available_rows
0,2026-03-01,275874,101910
1,2026-03-02,276269,103696
2,2026-03-03,311676,107362
3,2026-03-04,311675,109377
4,2026-03-05,311676,109740
5,2026-03-06,312187,110037
6,2026-03-07,312387,102153
7,2026-03-08,313374,101170
8,2026-03-09,313874,111313
9,2026-03-10,314047,113051


In [10]:
con.execute(f"""
    SELECT
        COUNT(*) AS client_page_pairs,
        COUNT(*) FILTER (
            WHERE available_days = 30
        ) AS pairs_with_full_30_days
    FROM (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) AS available_days
        FROM read_parquet('{march_path}')
        WHERE report_date BETWEEN DATE '2026-03-01'
                              AND DATE '2026-03-30'
        GROUP BY client_hash_id, content_hash_id
    )
""").df()

,client_page_pairs,pairs_with_full_30_days
0,331231,62182


In [11]:
con.execute(f"""
    WITH previous_period AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-02-28'
    ),
    pair_coverage AS (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(*) FILTER (
                WHERE gsc_data_available IS TRUE
            ) AS available_days
        FROM previous_period
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) AS client_page_pairs,
        COUNT(*) FILTER (
            WHERE available_days = 30
        ) AS pairs_with_full_30_days
    FROM pair_coverage
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_page_pairs,pairs_with_full_30_days
0,321546,54850


In [12]:
con.execute(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
    ),
    pair_coverage AS (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_available_days,
            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_available_days
        FROM daily
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) FILTER (
            WHERE previous_available_days = 30
        ) AS previous_complete,
        COUNT(*) FILTER (
            WHERE recent_available_days = 30
        ) AS recent_complete,
        COUNT(*) FILTER (
            WHERE previous_available_days = 30
              AND recent_available_days = 30
        ) AS complete_in_both
    FROM pair_coverage
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,previous_complete,recent_complete,complete_in_both
0,54850,62182,43245


In [13]:
con.execute(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available,
            gsc_impressions
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
    ),
    page_metrics AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_available_days,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_available_days,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_impressions,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_impressions

        FROM daily
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) AS eligible_pages,
        COUNT(*) FILTER (
            WHERE previous_impressions = 0
        ) AS zero_previous_impressions,
        COUNT(*) FILTER (
            WHERE recent_impressions = 0
        ) AS zero_recent_impressions,
        ROUND(AVG(previous_impressions), 2) AS avg_previous_impressions,
        ROUND(AVG(recent_impressions), 2) AS avg_recent_impressions
    FROM page_metrics
    WHERE previous_available_days = 30
      AND recent_available_days = 30
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,eligible_pages,zero_previous_impressions,zero_recent_impressions,avg_previous_impressions,avg_recent_impressions
0,43245,0,0,3450.72,4019.94


In [14]:
con.execute(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available,
            gsc_impressions
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
    ),
    page_metrics AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_days,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_days,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_impressions,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_impressions

        FROM daily
        GROUP BY client_hash_id, content_hash_id
    ),
    eligible AS (
        SELECT *,
            (previous_impressions - recent_impressions)
                / previous_impressions AS decline_rate
        FROM page_metrics
        WHERE previous_days = 30
          AND recent_days = 30
          AND previous_impressions > 0
    ),
    client_thresholds AS (
        SELECT *,
            quantile_cont(previous_impressions, 0.75)
                OVER (PARTITION BY client_hash_id) AS client_p75
        FROM eligible
    ),
    labelled AS (
        SELECT *,
            CASE
                WHEN decline_rate >= 0.30
                 AND previous_impressions >= client_p75
                THEN 1
                ELSE 0
            END AS review_candidate
        FROM client_thresholds
    )
    SELECT
        COUNT(*) AS eligible_pages,
        COUNT(DISTINCT client_hash_id) AS eligible_clients,
        SUM(review_candidate) AS positive_pages,
        ROUND(100.0 * AVG(review_candidate), 2) AS positive_percentage
    FROM labelled
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,eligible_pages,eligible_clients,positive_pages,positive_percentage
0,43245,23,1478.0,3.42


In [15]:
con.execute(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available,
            gsc_impressions
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
    ),
    page_metrics AS (
        SELECT
            client_hash_id,
            content_hash_id,
            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_days,
            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_days,
            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_impressions,
            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_impressions
        FROM daily
        GROUP BY client_hash_id, content_hash_id
    ),
    eligible AS (
        SELECT *,
            (previous_impressions - recent_impressions)
                / previous_impressions AS decline_rate
        FROM page_metrics
        WHERE previous_days = 30
          AND recent_days = 30
          AND previous_impressions > 0
    ),
    thresholds AS (
        SELECT *,
            quantile_cont(previous_impressions, 0.75)
                OVER (PARTITION BY client_hash_id) AS client_p75
        FROM eligible
    )
    SELECT
        client_hash_id,
        content_hash_id,
        previous_impressions,
        recent_impressions,
        ROUND(decline_rate * 100, 2) AS decline_percentage,
        client_p75
    FROM thresholds
    WHERE decline_rate >= 0.30
      AND previous_impressions >= client_p75
    ORDER BY decline_rate DESC
    LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,previous_impressions,recent_impressions,decline_percentage,client_p75
0,client_23a62021009f63c4,content_2ac8c7995de53cd1,92147.0,1815.0,98.03,5017.0
1,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,16819.0,542.0,96.78,3425.0
2,client_73cda7b4e4f265ea,content_29c1e9fb447ec17f,9022.0,330.0,96.34,3425.0
3,client_23a62021009f63c4,content_cf1af462f6317c31,30736.0,1134.0,96.31,5017.0
4,client_73cda7b4e4f265ea,content_0709f29e7f096e6d,55187.0,2246.0,95.93,3425.0
5,client_23a62021009f63c4,content_f173fa70b277cf04,8314.0,344.0,95.86,5017.0
6,client_23a62021009f63c4,content_73aa32cad383d8f9,14329.0,643.0,95.51,5017.0
7,client_73cda7b4e4f265ea,content_4bffcff197036709,3439.0,156.0,95.46,3425.0
8,client_23a62021009f63c4,content_04235ab203bbfd92,28564.0,1298.0,95.46,5017.0
9,client_23a62021009f63c4,content_b85c606ed4e64eb2,26125.0,1216.0,95.35,5017.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field classification — Lane 2

**Features (candidate predictive inputs):**
- Historical GSC impressions and clicks from a period before the prediction date.
- Historical average search position.
- Historical pageviews and sessions, only where GA4 data is available.
- Content metadata such as word count, content type, and content age.

**Label:**
- `review_candidate`: a historical proxy based on a 30% or greater decline in impressions and previous-period impressions at or above the client's 75th percentile.

**Context:**
- `client_hash_id`: identifies the client and supports client-level grouping.
- `content_hash_id`: identifies the content page.
- `report_date`: identifies the observation date.
- `gsc_data_available` and `ga4_data_available`: indicate whether the corresponding measurements are usable.

**Excluded:**
- `review_candidate` and its defining measurements from the predictive feature set, to prevent target leakage.
- IDs as predictive inputs, because they identify records rather than describe page performance.
- GA4 measurements where `ga4_data_available IS NOT TRUE`.
- June 2026 sealed test data during development.
- The April–June 2026 `fact_content_query_90d` table, because its fixed window occurs after the March development period.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:


con.execute(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS duplicate_count
    FROM read_parquet('{march_path}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


In [17]:


con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT report_date) AS distinct_dates
    FROM read_parquet('{march_path}')
""").df()

,total_rows,earliest_date,latest_date,distinct_dates
0,9841378,2026-03-01,2026-03-31,31


In [18]:


con.execute(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS both_available_rows
    FROM read_parquet('{march_path}')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


In [19]:
# Check complete GSC coverage across both comparison periods

con.execute(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
    ),
    pair_coverage AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_available_days,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_available_days

        FROM daily
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) FILTER (
            WHERE previous_available_days = 30
        ) AS previous_complete,

        COUNT(*) FILTER (
            WHERE recent_available_days = 30
        ) AS recent_complete,

        COUNT(*) FILTER (
            WHERE previous_available_days = 30
              AND recent_available_days = 30
        ) AS complete_in_both

    FROM pair_coverage
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,previous_complete,recent_complete,complete_in_both
0,54850,62182,43245


In [20]:
# Verify the historical review-candidate proxy distribution

con.execute(f"""
    WITH daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_data_available,
            gsc_impressions
        FROM read_parquet([
            '{warehouse}/fact_content_daily_performance/month=2026-01/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-02/*.parquet',
            '{warehouse}/fact_content_daily_performance/month=2026-03/*.parquet'
        ])
        WHERE report_date BETWEEN DATE '2026-01-30'
                              AND DATE '2026-03-30'
    ),
    page_metrics AS (
        SELECT
            client_hash_id,
            content_hash_id,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_days,

            COUNT(*) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_days,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-01-30'
                                      AND DATE '2026-02-28'
                  AND gsc_data_available IS TRUE
            ) AS previous_impressions,

            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-01'
                                      AND DATE '2026-03-30'
                  AND gsc_data_available IS TRUE
            ) AS recent_impressions

        FROM daily
        GROUP BY client_hash_id, content_hash_id
    ),
    eligible AS (
        SELECT *,
            (previous_impressions - recent_impressions)
                / previous_impressions AS decline_rate
        FROM page_metrics
        WHERE previous_days = 30
          AND recent_days = 30
          AND previous_impressions > 0
    ),
    thresholds AS (
        SELECT *,
            quantile_cont(previous_impressions, 0.75)
                OVER (PARTITION BY client_hash_id) AS client_p75
        FROM eligible
    ),
    labelled AS (
        SELECT *,
            CASE
                WHEN decline_rate >= 0.30
                 AND previous_impressions >= client_p75
                THEN 1
                ELSE 0
            END AS review_candidate
        FROM thresholds
    )
    SELECT
        COUNT(*) AS eligible_pages,
        COUNT(DISTINCT client_hash_id) AS eligible_clients,
        SUM(review_candidate) AS positive_pages,
        ROUND(100.0 * AVG(review_candidate), 2) AS positive_percentage
    FROM labelled
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,eligible_pages,eligible_clients,positive_pages,positive_percentage
0,43245,23,1478.0,3.42


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Limitations of the data

1. **Uneven history:** Clients have different amounts of historical data, so the warehouse does not provide equal coverage for every client.

2. **Missing analytics data:** GSC and GA4 availability varies across dates and clients. Missing values cannot automatically be treated as zero.

3. **Restricted eligible population:** Only 43,245 client-page pairs across 23 clients had complete GSC coverage in both comparison periods. The analysis therefore does not represent the entire warehouse.

4. **Proxy rather than ground truth:** A decline in impressions does not necessarily mean that content needs updating. The proxy identifies pages worth investigating, not pages confirmed to have content problems.

5. **Future-window limitation:** The query-level table covers 2 April–30 June 2026. It cannot be used as a source of March development features because it contains information from after the development period.

6. **No causal evidence:** Historical performance data can show that impressions changed, but cannot establish that editing a page will improve its performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.